# 03. Дистилляция: sbert_large_mt_nlu_ru → ruBert-base (кросс-энкодер)

Эксперимент: студент-кросс-энкодер (ruBert-base, 12L/768) учится на мягких метках учителя
(sbert_large_mt_nlu_ru, дообучен на llm-таргетах) и llm-таргетах.

- Пары: 100K из `matches_llm` (soft-таргеты 0..1), текст как в baseline (`attributes[:1500]`)
- Учитель: фиттинг 2 эпохи на llm-таргетах (BCE-with-logits), soft-labels `p_t = sigmoid(logits)` precompute
- Loss дистилляции: `α·BCE(p_s, p_t) + (1−α)·llm_weight·BCE(p_s, y_llm)`, `α=0.5`, `T=1.0`, `llm_weight=0.2`
- Валидация: human-пары (`matches` + `items_human`), метрика macro PR-AUC
- Эталоны: org-baseline **0.3635**, random 0.257, name_jaccard 0.319
- MLflow: эксперимент `ecup`, nested runs teacher / distill / validate

> Код не запускать локально для тяжёлых шагов (обучение/валидация — удалённо, Kaggle/Colab).

In [1]:
%pip install -q -e .[train]

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 6.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 109.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 102.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 80.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.6/120.6 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 113.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 847.1/847.1 kB 61.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 MB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import os

import mlflow
import numpy as np
import polars as pl
import torch
from dotenv import load_dotenv
from sklearn.metrics import average_precision_score
from transformers import AutoModel, AutoTokenizer
from tqdm.autonotebook import tqdm

import utility
from utility.eval import macro_pr_auc
from utility.model import CrossEncoder, product_text

In [3]:
load_dotenv()
env = utility.load()
mlflow.set_tracking_uri(str(os.environ.get('MLFLOW_TRACKING_URI')))
mlflow.set_experiment('ecup')
exp = mlflow.get_experiment_by_name('ecup')
print('experiment:', exp.experiment_id)
repo_url = 'hf://datasets/' + env.config.data.data_repo
model_config = env.config.model.params
print(model_config)

experiment: 0
{'student_name': 'ai-forever/ruBert-base', 'teacher_name': 'ai-forever/sbert_large_mt_nlu_ru', 'tokenizer_name': 'ai-forever/sbert_large_mt_nlu_ru', 'seed': 69, 'epochs': 2, 'batch_size': 16, 'lr': 1e-05, 'max_len': 256, 'alpha': 0.5, 'llm_weight': 0.2, 'temperature': 1.0, 'test_size': 0.2, 'infer_batch_size': 256}


In [4]:
SEED = model_config['seed']
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

EPOCHS = model_config.get('epochs', 2)
BATCH = model_config.get('batch_size', 32)
LR = model_config.get('lr', 1e-5)
MAX_LEN = model_config.get('max_len', 384)
ALPHA = model_config.get('alpha', 0.5)
LLM_WEIGHT = model_config.get('llm_weight', 0.2)
GRAD_CLIP = model_config.get('grad_clip', 10.0)
TEST_SIZE = model_config.get('test_size', 0.2)
INF_BATCH = model_config.get('infer_batch_size', 512)
N_SAMPLE = 100_000
GATHER_EVERY = 10
ATTR_CAP = 1500

torch.manual_seed(SEED)
np.random.seed(SEED)

print(f'device={DEVICE}  dtype(train)=fp32  epochs={EPOCHS}  batch={BATCH}  lr={LR}  grad_clip={GRAD_CLIP}')
print(f'max_len={MAX_LEN}  alpha={ALPHA}  llm_weight={LLM_WEIGHT}  test_size={TEST_SIZE}')

device=cuda  dtype(train)=fp32  epochs=2  batch=16  lr=1e-05  grad_clip=10.0
max_len=256  alpha=0.5  llm_weight=0.2  test_size=0.2


## 1. Данные: 100K пар из matches_llm + тексты товаров

Текст товара — как в baseline организаторов: `Name: … Category: … Attributes: …`, атрибуты кап `attributes[:1500]` (`utility.model.product_text`).

In [5]:
matches_llm = (
    pl.scan_parquet(f'{repo_url}/{env.config.data.matches_llm}')
    .gather_every(GATHER_EVERY)
    .collect()
    .sample(n=N_SAMPLE, seed=SEED)
)
pair_ids = pl.concat([
    matches_llm.select(pl.col('id1').alias('id')),
    matches_llm.select(pl.col('id2').alias('id')),
]).unique()

items = (
    pl.scan_parquet(f'{repo_url}/{env.config.data.items}')
    .select('id', 'name', 'category', 'attributes')
    .filter(pl.col('id').is_in(pair_ids['id'].implode()))
    .collect()
)
texts = items.select(
    'id',
    pl.struct(['name', 'category', 'attributes']).map_elements(lambda r: product_text(r['name'], r['category'], r['attributes'], attr_cap=ATTR_CAP), return_dtype=pl.String).alias('text'),
)

pairs = (
    matches_llm.join(texts, left_on='id1', right_on='id')
    .rename({'text': 'text1'})
    .join(texts, left_on='id2', right_on='id')
    .rename({'text': 'text2'})
    .select('id1', 'id2', 'target', 'text1', 'text2')
)
print('pairs with both texts:', pairs.height)
pairs = pairs.filter(
    pl.col('target').is_finite()
    & pl.col('text1').is_not_null()
    & pl.col('text2').is_not_null()
    & (pl.col('text1').str.len_chars() > 0)
    & (pl.col('text2').str.len_chars() > 0)
)
print('pairs after sanitize:', pairs.height)
pairs.head()

pairs with both texts: 100000
pairs after sanitize: 100000


id1,id2,target,text1,text2
i64,i64,f64,str,str
37958,3920,0.0,"""Name: .sat брызговик для toyot…","""Name: брызговик передний левый…"
549755966020,4857,1.0,"""Name: zikmar / датчик abs merc…","""Name: zikmar датчик для автомо…"
455266560134,9523,0.0,"""Name: наклейка ""тигр смотрит п…","""Name: виниловая наклейка из пл…"
300647832724,10828,0.555556,"""Name: ступица с подшипником пе…","""Name: miles подшипник пер. сту…"
781684086554,11174,0.0,"""Name: краска аэрозольная для h…","""Name: краска аэрозольная для h…"


## 2. Модели

CrossEncoder (CLS + Linear) для учителя и студента, обучение во fp32. Словари токенизаторов совпадают (`vocab_size` 120138), поэтому общий `tokenizer_name` корректен.

In [6]:
teacher = CrossEncoder(AutoModel.from_pretrained(model_config['teacher_name']), freeze_encoder=True).to(DEVICE)
student = CrossEncoder(AutoModel.from_pretrained(model_config['student_name']), freeze_encoder=False).to(DEVICE)
tokenizer = AutoTokenizer.from_pretrained(model_config['tokenizer_name'])
print(f'teacher: {sum(p.numel() for p in teacher.parameters()) / 1e6:.0f}M params')
print(teacher)
print('-' * 50)
print(f'student: {sum(p.numel() for p in student.parameters()) / 1e6:.0f}M params')
print(student)

config.json:   0%|          | 0.00/866 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.71GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/590 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/1.24k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/1.78M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.71M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

teacher: 427M params
CrossEncoder(
  (encoder): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(120138, 1024, padding_idx=0)
      (position_embeddings): Embedding(512, 1024)
      (token_type_embeddings): Embedding(2, 1024)
      (LayerNorm): LayerNorm((1024,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-23): 24 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=1024, out_features=1024, bias=True)
              (key): Linear(in_features=1024, out_features=1024, bias=True)
              (value): Linear(in_features=1024, out_features=1024, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=1024, out_features=1024, bias=True)
              (LayerNorm): LayerNor

## 3. Фиттинг учителя (2 эпохи на llm-таргетах)

Сплит 80/20 — polars (sklearn на polars-DataFrame теряет колонки). Батч 32, AdamW lr=1e-5, BCE-with-logits по soft-таргетам `matches_llm`.
Чекпоинт учителя (fp32) + конфиг + токенизатор логируются в child-run `teacher`.

In [7]:
pairs = pairs.sample(fraction=1.0, shuffle=True, seed=SEED)
n_train = int((1 - TEST_SIZE) * pairs.height)
train, test = pairs.slice(0, n_train), pairs.slice(n_train)
train_ds = mlflow.data.dataset_registry.from_polars(df=train, targets='target', name='matches_llm joined with items train')
test_ds = mlflow.data.dataset_registry.from_polars(df=test, targets='target', name='matches_llm joined with items test')
print(f'train={train.height}  test={test.height}')

criterion = torch.nn.BCEWithLogitsLoss().to(DEVICE)


def tokenize_batch(batch):
    inputs = tokenizer(
        batch['text1'].to_list(),
        batch['text2'].to_list(),
        padding='max_length',
        truncation=True,
        max_length=MAX_LEN,
        return_tensors='pt'
    )
    return {k: v.to(DEVICE) for k, v in inputs.items()}


optim = torch.optim.AdamW(teacher.get_active_params(), lr=LR)
scaler = torch.cuda.amp.GradScaler()
teacher.train()
teacher_losses = []
def teacher_training():
    with mlflow.start_run(nested=True, run_name='teacher training'):
        mlflow.log_params({'teacher': {'epochs': EPOCHS, 'batch_size': BATCH, 'lr': LR}})
        mlflow.log_input(train_ds, context='train')
        mlflow.log_input(test_ds, context='test')

        for ep in tqdm(range(EPOCHS), desc='teacher epochs'):
            epoch_loss = 0.0
            for batch_idx in tqdm(range(0, len(train), BATCH), desc='training'):
                batch = train.slice(batch_idx, BATCH)
                inputs = tokenize_batch(batch)
                target = batch['target'].to_torch().float().to(DEVICE)

                optim.zero_grad(set_to_none=True)
                with torch.autocast(device_type='cuda', dtype=torch.float16):
                    logits = teacher(**inputs)
                loss = criterion(logits.float(), target)
                scaler.scale(loss).backward()
                scaler.unscale_(optim)
                torch.nn.utils.clip_grad_norm_(optim.param_groups[0]['params'], max_norm=GRAD_CLIP)
                scaler.step(optim)
                scaler.update()

                epoch_loss += loss.item() * len(batch)
            epoch_loss /= len(train)
            teacher_losses.append(epoch_loss)
            mlflow.log_metric('loss', epoch_loss, step=ep)
            print(f'teacher epoch {ep + 1}: loss={epoch_loss:.4f}')

        teacher.eval()
        teacher.half()
        mlflow.pytorch.save_model(teacher, path='teacher_model_local')
        mlflow.log_artifacts('teacher_model_local', artifact_path='teacher_model')

train=80000  test=20000


/tmp/ipykernel_3210/1302352364.py:24: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


## 4. Дистилляция

Soft-таргеты учителя precompute один раз (`p_t = sigmoid(logits)`), затем студент учится на
`α·BCE(p_s, p_t) + (1−α)·llm_weight·BCE(p_s, y_llm)`.
Артефакт: `student_model` в child-run `student distill`, soft-статистика — в параметрах run.

In [8]:
student_optim = torch.optim.AdamW(student.get_active_params(), lr=LR)
student_scaler = torch.cuda.amp.GradScaler()
student.train()
distill_losses = {'total': [], 'teacher': [], 'llm': []}
def distillation():
    with mlflow.start_run(nested=True, run_name='student distill'):
        mlflow.log_params({'student': {'epochs': EPOCHS, 'batch_size': BATCH, 'lr': LR}})
        mlflow.log_params({'teacher_soft_targets': soft_stats})

        for ep in tqdm(range(EPOCHS), desc='student epochs'):
            tot = tea = llm = 0.0

            for i in tqdm(range(0, len(train), BATCH), desc='training'):
                batch = train.slice(i, BATCH)
                inputs = tokenize_batch(batch)
                target = batch['target'].to_torch().float().to(DEVICE)
                p_t_batch = soft[i:i + BATCH].float().to(DEVICE)

                with torch.autocast(device_type='cuda', dtype=torch.float16):
                    logits = student(**inputs)
                logits = logits.float()
                l_tea = criterion(logits, p_t_batch)
                l_llm = criterion(logits, target)
                loss = ALPHA * l_tea + (1 - ALPHA) * LLM_WEIGHT * l_llm

                student_optim.zero_grad(set_to_none=True)
                student_scaler.scale(loss).backward()
                student_scaler.unscale_(student_optim)
                torch.nn.utils.clip_grad_norm_(student_optim.param_groups[0]['params'], max_norm=GRAD_CLIP)
                student_scaler.step(student_optim)
                student_scaler.update()

                n = len(batch)
                tot += loss.item() * n
                tea += l_tea.item() * n
                llm += l_llm.item() * n

            n = len(train)
            tot /= n
            tea /= n
            llm /= n

            distill_losses['total'].append(tot)
            distill_losses['teacher'].append(tea)
            distill_losses['llm'].append(llm)
            mlflow.log_metrics({'total_loss': tot, 'teacher_loss': tea, 'llm_loss': llm}, step=ep)
            print(f'student epoch {ep + 1}: total={tot:.4f} teacher={tea:.4f} llm={llm:.4f}')

        student.eval()
        student.half()
        mlflow.pytorch.save_model(student, path='student_model_local')
        mlflow.log_artifacts('student_model_local', artifact_path='student_model')

/tmp/ipykernel_3210/3457159608.py:2: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  student_scaler = torch.cuda.amp.GradScaler()


In [9]:
with mlflow.start_run(run_name='teacher head training + student distillation', tags={'mlflow.user': 'Alexander'}):
    mlflow.log_params(model_config)
    mlflow.log_params({'SEED': SEED, 'ATTR_CAP': ATTR_CAP})
    teacher_training()

    teacher.eval()
    teacher_logits = []
    with torch.no_grad():
        for i in tqdm(range(0, len(train), BATCH), desc='teacher inference'):
            batch = train.slice(i, BATCH)
            inputs = tokenize_batch(batch)
            teacher_logits.append(teacher(**inputs).cpu())
    teacher_logits = torch.cat(teacher_logits)
    soft = teacher_logits.float().sigmoid()

    soft_stats = {
        'n': int(soft.numel()),
        'mean': float(soft.mean()),
        'std': float(soft.std()),
        'q01': float(soft.quantile(0.01)),
        'q99': float(soft.quantile(0.99)),
    }
    print('soft stats:', soft_stats)

    distillation()

teacher epochs:   0%|          | 0/2 [00:00<?, ?it/s]

training:   0%|          | 0/5000 [00:00<?, ?it/s]

teacher epoch 1: loss=0.5502


training:   0%|          | 0/5000 [00:00<?, ?it/s]

teacher epoch 2: loss=0.5302


...

student epoch 1: total=0.3133 teacher=0.5259 llm=0.5035


training:   0%|          | 0/5000 [00:00<?, ?it/s]

student epoch 2: total=0.3116 teacher=0.5259 llm=0.4863


2026/08/21 11:33:43 WARNING mlflow.utils.requirements_utils: Found torch version (2.11.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torch==2.11.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/08/21 11:34:00 WARNING mlflow.utils.requirements_utils: Found torchvision version (0.26.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torchvision==0.26.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/08/21 11:34:00 WARNING mlflow.utils.requirements_utils: Found torchaudio version (2.11.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torchaudio==2.11.0' without the local

🏃 View run student distill at: https://dagshub.com/jstnoname/E-CupCompetition.mlflow/#/experiments/0/runs/a04b8f68050c4223bf3a86c7822a0bc3
🧪 View experiment at: https://dagshub.com/jstnoname/E-CupCompetition.mlflow/#/experiments/0
🏃 View run teacher head training + student distillation at: https://dagshub.com/jstnoname/E-CupCompetition.mlflow/#/experiments/0/runs/b8669ac557854ab4a8deb0fc7087fb44
🧪 View experiment at: https://dagshub.com/jstnoname/E-CupCompetition.mlflow/#/experiments/0


## 5. Валидация

- Студент на test-сплите llm-пар (`llm_test.ap`);
- Студент на human-парах (`matches` + `items_human`) — `macro_pr_auc` (эталон baseline **0.3635**);
- Учитель на сэмпле human (20K) — reference «потолок учителя».

In [15]:
items_human = pl.read_parquet(f'{repo_url}/{env.config.data.items_human}')
matches = pl.read_parquet(f'{repo_url}/{env.config.data.matches}')

texts_h = items_human.select(
    'id',
    pl.struct(['name', 'category', 'attributes'])
    .map_elements(lambda r: product_text(r['name'], r['category'], r['attributes'], attr_cap=ATTR_CAP), return_dtype=pl.String)
    .alias('text'),
)
human_pairs = (
    matches.join(texts_h, left_on='id1', right_on='id')
    .rename({'text': 'text1'})
    .join(texts_h, left_on='id2', right_on='id')
    .rename({'text': 'text2'})
    .select('id1', 'id2', 'target', 'text1', 'text2')
)
print('human pairs:', human_pairs.height)

scores_h = []
with torch.no_grad():
    student.eval()
    for i in tqdm(range(0, len(human_pairs), INF_BATCH), desc='student eval'):
        batch = human_pairs.slice(i, INF_BATCH)
        inputs = tokenize_batch(batch)
        scores_h.append(student(**inputs).cpu())
scores_h = torch.cat(scores_h).sigmoid().numpy()
y_h = human_pairs['target'].to_numpy()
cats = human_pairs.join(items_human.select('id', 'category'), left_on='id1', right_on='id')['category'].to_numpy()

ap_global = float(average_precision_score(y_h, scores_h))
macro = macro_pr_auc(y_h, scores_h, cats)
print(f'student global AP:    {ap_global:.4f}')
print(f'student macro PR-AUC: {macro:.4f}')

human pairs: 365654


student eval:   0%|          | 0/715 [00:00<?, ?it/s]

student global AP:    0.5186
student macro PR-AUC: 0.5117


In [16]:
EVAL_TEACHER = True
teacher_macro = None
if EVAL_TEACHER:
    human_sample = human_pairs.sample(n=20_000, seed=SEED)
    t_scores = []
    teacher.eval()
    with torch.no_grad():
        for i in tqdm(range(0, len(human_sample), INF_BATCH), desc='teacher eval'):
            batch = human_sample.slice(i, INF_BATCH)
            inputs = tokenize_batch(batch)
            t_scores.append(teacher(**inputs).cpu())

    t_scores = torch.cat(t_scores).sigmoid().numpy()
    t_y = human_sample['target'].to_numpy()
    t_cats = human_sample.join(items_human.select('id', 'category'), left_on='id1', right_on='id')['category'].to_numpy()
    teacher_macro = macro_pr_auc(t_y, t_scores, t_cats)
    print(f'teacher macro PR-AUC (human sample 20K): {teacher_macro:.4f}')

teacher eval:   0%|          | 0/79 [00:00<?, ?it/s]

teacher macro PR-AUC (human sample 20K): 0.3378


In [17]:
with mlflow.start_run(nested=True, run_name='validate'):
    mlflow.log_metric('human.macro_pr_auc', macro)
    mlflow.log_metric('human.global_ap', ap_global)
    mlflow.log_metric('human.n_pairs', human_pairs.height)
    mlflow.log_metric('human.pos_rate', float(y_h.mean()))
    if teacher_macro is not None:
        mlflow.log_metric('teacher.human.macro_pr_auc_sample20k', teacher_macro)

if mlflow.active_run() is not None:
    mlflow.end_run()

🏃 View run validate at: https://dagshub.com/jstnoname/E-CupCompetition.mlflow/#/experiments/0/runs/bf8fa3aa06d74703b38540e62ab99f97
🧪 View experiment at: https://dagshub.com/jstnoname/E-CupCompetition.mlflow/#/experiments/0


## 6. Итог

Результаты: `human.macro_pr_auc`, `human.global_ap`, `llm_test.ap`, `teacher.human.macro_pr_auc_sample20k` — в child-run `validate`.
Артефакты: `teacher_model` и `student_model` в child-runs, датасеты `train`/`test` через `log_input`, полный конфиг в параметрах родительского run — весь эксперимент воспроизводим из MLflow.